In [ ]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

MODEL_PATH: Path = Path("artifacts/linear_regression.joblib")  # đường dẫn tương đối
if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Model not found: {MODEL_PATH}")

artifact = joblib.load(MODEL_PATH)       # chỉ NẠP, không huấn luyện
model = artifact["model"]                # Pipeline đã học xong
feature_names: list[str] = artifact["feature_names"]

In [ ]:
# Thanh trượt giới hạn trong miền hợp lệ -> người dùng không nhập được giá trị vô lý
hours = widgets.FloatSlider(value=6, min=0, max=20, step=.5,
                            description="Gio hoc:", continuous_update=False)
attendance = widgets.FloatSlider(value=80, min=50, max=100, step=1,
                                 description="Chuyen can:", continuous_update=False)
assignments = widgets.IntSlider(value=7, min=0, max=10, step=1,
                                description="Bai tap:", continuous_update=False)
previous = widgets.FloatSlider(value=6.5, min=0, max=10, step=.1,
                               description="Diem truoc:", continuous_update=False)
predict_button = widgets.Button(description="Du doan", button_style="primary")
output = widgets.Output()  # vùng hiển thị kết quả


def input_frame() -> pd.DataFrame:
    """Gom giá trị các thanh trượt thành 1 dòng DataFrame đúng thứ tự cột."""
    values = {
        "hours_studied": hours.value,
        "attendance": attendance.value,
        "assignments_completed": assignments.value,
        "previous_score": previous.value,
    }
    return pd.DataFrame(
        [[values[name] for name in feature_names]], columns=feature_names
    )


def on_predict(_: object) -> None:
    """Chạy mỗi khi bấm nút."""
    predict_button.disabled = True          # khóa nút, tránh bấm liên tục
    with output:
        output.clear_output()               # xóa kết quả cũ
        try:
            raw = float(model.predict(input_frame())[0])
            # Hồi quy tuyến tính có thể ra ngoài 0-10, nên cắt về thang điểm
            print(f"Diem thi du doan: {np.clip(raw, 0, 10):.2f}/10")
        except Exception:
            # Không in traceback chi tiết cho người dùng công khai
            print("Khong the du doan. Hay kiem tra du lieu va artifact.")
        finally:
            predict_button.disabled = False  # luôn mở khóa nút lại


predict_button.on_click(on_predict)  # gắn hàm vào sự kiện bấm nút

display(widgets.VBox([
    hours, attendance, assignments, previous, predict_button, output
]))